# trails.ipynb

This is a placeholder for the trails.ipynb Jupyter notebook.

In [ ]:
print("ok")

: 

In [ ]:
%pip install langchain-experimental==0.3.2
%pip install -e .

Load The PDF 


In [ ]:
def load_pdf_file(file_path):
    from langchain.document_loaders import PyPDFLoader, DirectoryLoader
    loader = DirectoryLoader(
        file_path,
        glob="**/*.pdf",
        loader_cls=PyPDFLoader,
        recursive=True,
        show_progress=True
    )
    return loader.load()


In [ ]:
%pip install tqdm
%pip install pypdf

extract = load_pdf_file("data/")

Split The pdf into clunks

In [ ]:
def text_splitter(extract):
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
        length_function=len
    )
    return text_splitter.split_documents(extract)

In [ ]:
# Extract text from each Document and split it
text_chunks = text_splitter(extract)
print(f"Total number of text chunks: {len(text_chunks)}")

Download the embedding from hugging face
    

In [ ]:
def download_huggingface_embedding():
    from langchain.embeddings import HuggingFaceEmbeddings
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
     
    )

In [ ]:
%pip install sentence-transformers

embedding = download_huggingface_embedding()

In [ ]:
query_results = embedding.embed_query("Hello world this is a test query")
print(f"Query embedding: {len(query_results)} dimensions")

In [ ]:
%pip install pinecone 

In [ ]:
from pinecone import Pinecone, ServerlessSpec
pc= Pinecone("pcsk_UVJyr_KwbiHLcm9W7kM8Yd6f6m4tzZLxVqdkJp9UYBgEUGbfX8UpyPEE6zp5oZnPpbgZg", environment="us-west1-gcp")

index_name = "test"

if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        dimension=384,
        metric="cosine",
        cloud="aws",
        region="us-east-1",
        embed={
            "model": "llama-text-embed-v2",
            "field_map": {"text": "chunk_text"}
        }
    )

In [ ]:
%pip install langchain-pinecone

import os
os.environ["PINECONE_API_KEY"] = "pcsk_UVJyr_KwbiHLcm9W7kM8Yd6f6m4tzZLxVqdkJp9UYBgEUGbfX8UpyPEE6zp5oZnPpbgZg"

from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    text_chunks,
    embedding=embedding,
    index_name=index_name,
    pinecone_api_key="pcsk_UVJyr_KwbiHLcm9W7kM8Yd6f6m4tzZLxVqdkJp9UYBgEUGbfX8UpyPEE6zp5oZnPpbgZg"
)

In [ ]:
retriever= docsearch.as_retriever(
    search_type="similarity",search_kwargs={
        "k": 3})
def query_retriever(query):
    results = retriever.invoke(query)
    return results
query = "What is the purpose of the CureAI project?"
results = query_retriever(query)
for result in results:
    print(f"Document: {result.metadata['source']}")
    print(f"Text: {result.page_content}\n")

In [ ]:
query = "What is the Acne?"
results = query_retriever(query)
print(f"Query: {query}")
for result in results:
    print(f"Document: {result.metadata['source']}")
    print(f"Text: {result.page_content}\n")


In [ ]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from openai import OpenAI
from langchain_core.messages import HumanMessage

# Load environment variables from .env file
load_dotenv()

# Retrieve the API key from environment variables
api_key = os.getenv("OPENAI_API_KEY")

# Check if the API key is set
if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable not set")

# Initialize the OpenAI client with the API key
client = OpenAI(api_key=api_key)

# Example: Make a simple OpenAI API call
try:
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": "Hello, world!"}]
    )
    print("OpenAI Response:", response.choices[0].message.content)
except Exception as e:
    print(f"OpenAI Error: {e}")

# Initialize the LangChain ChatOpenAI client
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.7,
    max_tokens=1000,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0,
    api_key=api_key  # Explicitly pass the API key
)

# Example: Make a simple LangChain API call
try:
    messages = [HumanMessage(content="Hello from LangChain!")]
    response = llm.invoke(messages)
    print("LangChain Response:", response.content)
except Exception as e:
    print(f"LangChain Error: {e}")